# Customer Churn Prediction — End-to-End Data Science Project

**Dataset:** Telco Customer Churn (Kaggle)  
**Goal:** Predict which customers are likely to churn and identify key drivers.  
**Stack:** pandas · scikit-learn · XGBoost · SHAP · Streamlit

---
## Table of Contents
1. Business Understanding
2. Data Loading & Quality Check
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Modelling & Model Selection
6. Evaluation (beyond accuracy)
7. Explainability with SHAP
8. Business Recommendations


## 1. Business Understanding

Customer churn — when a customer stops using a service — is expensive.  
Acquiring a new customer costs **5–7× more** than retaining an existing one.

**Business questions we're answering:**
- Which customers are most at risk of churning?
- What are the top drivers of churn?
- What targeted actions can reduce churn?

**Success metric:** ROC-AUC (we care about ranking risk, not just raw accuracy)  
**Secondary:** Recall for churners — we'd rather flag a non-churner than miss a churner.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay, PrecisionRecallDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('Libraries loaded ✓')

## 2. Data Loading & Quality Check

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.columns

In [ ]:
# Data types & missing values
info = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique()
})
info[info['null_count'] > 0]  # Show only columns with nulls

In [ ]:
# Fix TotalCharges — loaded as object due to blank strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_dropped = df['TotalCharges'].isnull().sum()
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f'Dropped {n_dropped} rows with missing TotalCharges')
print(f'Churn rate: {df["Churn"].mean():.1%}  — class imbalance to handle!')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA — Customer Churn Drivers', fontsize=16, fontweight='bold')

# 1. Churn rate
churn_pct = df['Churn'].value_counts(normalize=True)
axes[0,0].pie(churn_pct, labels=['No Churn', 'Churned'], autopct='%1.1f%%',
              colors=['#4CAF50', '#F44336'], startangle=90)
axes[0,0].set_title('Overall Churn Rate')

# 2. Tenure
df[df['Churn']==0]['tenure'].hist(ax=axes[0,1], alpha=0.7, bins=30, label='No Churn', color='#4CAF50')
df[df['Churn']==1]['tenure'].hist(ax=axes[0,1], alpha=0.7, bins=30, label='Churned', color='#F44336')
axes[0,1].set_title('Tenure Distribution')
axes[0,1].legend()

# 3. Monthly Charges
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0,2])
axes[0,2].set_title('Monthly Charges by Churn')
axes[0,2].set_xlabel('Churn')
plt.sca(axes[0,2]); plt.title('Monthly Charges by Churn')

# 4. Contract type
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
contract_churn.plot(kind='bar', ax=axes[1,0], color='#2196F3', edgecolor='white')
axes[1,0].set_title('Churn Rate by Contract Type')
axes[1,0].set_ylabel('Churn Rate')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=30)

# 5. Internet service
internet_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
internet_churn.plot(kind='bar', ax=axes[1,1], color='#9C27B0', edgecolor='white')
axes[1,1].set_title('Churn Rate by Internet Service')
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=30)

# 6. Payment method
payment_churn = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
payment_churn.plot(kind='bar', ax=axes[1,2], color='#FF9800', edgecolor='white')
axes[1,2].set_title('Churn Rate by Payment Method')
axes[1,2].set_xticklabels(axes[1,2].get_xticklabels(), rotation=40, ha='right')

plt.tight_layout()
#plt.savefig('outputs/eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Key EDA Insights:
  • ~26% churn rate → significant class imbalance
  • Short-tenure customers churn most (first 12 months are critical)
  • Month-to-month contracts have ~43% churn vs 11% for 1-year
  • Fiber optic users churn more — possibly pricing sensitivity
  • Electronic check users churn significantly more
""")

## 4. Feature Engineering

> Good features often matter more than model choice. Here we create business-driven features.

In [ ]:
df_fe = df.copy()

# Average monthly charge over customer lifetime
df_fe['AvgMonthlyCharge'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# Number of services subscribed to (more services = stickier)
service_cols = ['PhoneService','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df_fe['ServiceCount'] = df_fe[service_cols].apply(lambda col: (col == 'Yes').astype(int)).sum(axis=1)

# Tenure bucket
df_fe['TenureGroup'] = pd.cut(
    df_fe['tenure'], bins=[0, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4+yr']
)

# High-risk flag: month-to-month + fiber + high charges
df_fe['HighRiskFlag'] = (
    (df_fe['Contract'] == 'Month-to-month') &
    (df_fe['InternetService'] == 'Fiber optic') &
    (df_fe['MonthlyCharges'] > df_fe['MonthlyCharges'].median())
).astype(int)

print(f'High-risk customers: {df_fe["HighRiskFlag"].sum():,}')
print(f'High-risk churn rate: {df_fe.groupby("HighRiskFlag")["Churn"].mean()[1]:.1%}')
df_fe[['AvgMonthlyCharge','ServiceCount','TenureGroup','HighRiskFlag']].describe()

In [ ]:
df_fe

In [ ]:
# Encode categoricals
df_model = df_fe.copy()
y = df_model.pop('Churn')

for col in df_model.select_dtypes(include=['object', 'category']).columns:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

X = df_model.values
feature_names = df_model.columns.tolist()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Handle class imbalance with SMOTE
sm = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

In [ ]:
df_model

## 5. Modelling & Model Selection

We compare three models:
- **Logistic Regression** — interpretable baseline
- **Random Forest** — handles non-linearity well
- **XGBoost** — typically best for tabular data

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05,
                                          max_depth=5, eval_metric='logloss',
                                          random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_sc, y_train_sm, cv=cv, scoring='roc_auc')
    model.fit(X_train_sc, y_train_sm)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)

    results[name] = dict(model=model, cv_auc=cv_scores.mean(),
                         cv_std=cv_scores.std(), test_auc=test_auc,
                         y_pred=y_pred, y_proba=y_proba)

    print(f'{name:25s} | CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test AUC: {test_auc:.4f}')

best_name = max(results, key=lambda n: results[n]['test_auc'])
print(f'\n✓ Best model: {best_name}')

## 6. Evaluation (Beyond Accuracy)

In [ ]:
best = results[best_name]

print(f'=== {best_name} ===')
print(classification_report(y_test, best['y_pred'], target_names=['No Churn', 'Churned']))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'{best_name} — Evaluation', fontsize=14, fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, best['y_pred'], display_labels=['No Churn','Churned'],
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(y_test, best['y_proba'], ax=axes[1])
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_title('ROC Curve')

PrecisionRecallDisplay.from_predictions(y_test, best['y_proba'], ax=axes[2])
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout()
#plt.savefig('outputs/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Explainability with SHAP

> **Why SHAP?** Stakeholders need to understand *why* a customer was flagged — not just that they were. SHAP bridges the gap between model performance and business trust.

In [ ]:
best_model = results[best_name]['model']

explainer   = shap.Explainer(best_model, X_test_sc)
shap_values = explainer(X_test_sc)

# Global feature importance
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names, show=False)
plt.title('SHAP — Feature Impact on Churn Probability')
plt.tight_layout()
#plt.savefig('outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall plot for a single high-risk customer
churned_idx = np.where(y_test.values == 1)[0][0]
shap.waterfall_plot(shap_values[churned_idx], max_display=12)
plt.title('Why this customer is high-risk')
plt.tight_layout()
plt.show()

## 8. Business Recommendations

Based on our model's findings:

| Driver | Finding | Recommended Action |
|---|---|---|
| **Contract type** | Month-to-month customers churn 43% | Offer incentives to switch to annual contracts |
| **Tenure < 12 months** | First year is highest risk | Onboarding check-in at 30/90 days |
| **Fiber optic + high charges** | Price sensitivity signal | Bundle discount or loyalty pricing |
| **Electronic check** | Correlated with churn | Nudge toward auto-pay with small discount |
| **No tech support** | Higher churn rate | Offer free trial of tech support for at-risk customers |
